### Step 1: Import Libraries & API Keys

In [8]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json
import requests

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

sk-proj-


### Step 2: Set up Pushover

In [2]:
# Step 2a -> Setu up account in your browser
# Step 2b -> Set up the app on your phone
# Step 2c -> In the browser create an "Application/API Token"
# Step 2d -> Add the User Key and API token to your .env file
# Step 2e -> Install the Pushover app on your phone and log in with the same account
# Step 2f -> Run the code below to send a test notification to your phone

load_dotenv()

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


### Step 3: Test Pushover

In [ ]:
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [8]:
send_notification("Hello from the AI Engineering course!")

### Step 4: Describe Pushover as an LLM tool

In [4]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a notification to the user's phone using the Pushover service.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
            }
        },
        "required": ["message"]
    }
}

### Step 5: Add Pushover to the list of tools for the LLM

In [5]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 6: Calling the tool from an LLM

In [19]:
def handle_tool_call(tool_calls):
    # Return what to out context about tool call results, a dictionary
    tool_call = tool_calls[0]
    args = json.loads(tool_call.function.arguments)

    # Actually send the notification, i.e. call the tool
    send_notification(args["message"])
    #print(f"Sent notification with message: {args['message']}")
    tool_call_result = {
        "role": "tool",
        "content": f"Notification sent: {args['message']}",
        "tool_call_id": tool_call.id
    }
    # Return what to add to the context about tool call results, a dictionary
    return tool_call_result


In [20]:
client = OpenAI(api_key=OPENAI_API_KEY)

messages=[
    {"role": "user", "content": "Send a notification to my phone that says telling me what amazing progress I'm making on the AI Engineering course!"}
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message

# Check if model wants to call a tool
if message.tool_calls:
    # Handle the tool call
    tool_result = handle_tool_call(message.tool_calls) # Whole list of tool calls
    # Add message to context, i.e. messages
    messages.append(message)
    # Add Info about tool call response to the message content
    messages.append(tool_result)
    # Invoke the LLM one more time to get its update response
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    message = response.choices[0].message

print(message.content)

I've sent a notification to your phone telling you what amazing progress you're making on the AI Engineering course! Keep it up!


In [14]:
#if message.tool_calls:
#    tool_call = message.tool_calls[0]
#    args = json.loads(tool_call.function.arguments)

    # Actually send the notification
#    send_notification(args["message"])
#    print(f"Sent notification with message: {args['message']}")
#else:
#    print(message.content)